## Code to test strategies for LTC self-insurance
- [X] Import history of key funds/indexes: S&P 500, DJIA, Money Market
- [X] Build Simulation based on random date forward
- [O] Build Simulation based on Joint Distribution of Index and Money Market
- [O] Build Simulation using 90/10 or 85/15 strategy 
- [O] 
- [O] 


In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time,  datetime
from datetime import date
from ta.momentum import RSIIndicator
import yfinance as yf
import os.path
from fredapi import Fred
import math
import yaml

with open('fred_api.yaml', 'r') as file:
    configs = yaml.safe_load(file)

fred_api_key = configs['fred_api']


In [2]:
def get_prices(ticker, start="1999-01-01", end=None, printname=False):
    """
    Given a stock ticker, this function uses the yfinance API to pull prices.
    Optionally prints the company long name using get_info().
    """

    if end is None:
        end = datetime.date.today().strftime("%Y-%m-%d")

    # Print company name safely
    if printname and isinstance(ticker, str):
        try:
            compdata = yf.Ticker(ticker)
            info = compdata.get_info()
            company_name = info.get("longName", "Name not available")
            print(f"Getting {ticker}: {company_name}")
        except Exception as e:
            print(f"Getting {ticker}: (Name unavailable)")

    try:
        prices = yf.download(
            ticker, start=start, end=end, progress=False, interval="1mo"
        )

        if prices.empty:
            raise ValueError("No price data returned.")

        # Handle MultiIndex safely
        if isinstance(prices.columns, pd.MultiIndex):
            prices = prices["Close"]
        else:
            prices = prices["Close"]

        return prices

    except Exception as e:
        print(f"Error retrieving price data for {ticker}: {e}")
        return None

In [3]:
start_date = "1929-01-01"
start_date = "1947-01-01"
end_date = datetime.date.today().strftime("%Y-%m-%d")

indexes = {
    # "^DJI": "Dow Jones Industrial Average",
    "^GSPC": "Standard & Poors 500",
    # "^IRX": "13-week Treasury Bill yield",
    # "^IXIC": "NASDAQ Composite,",
    # "^RUT": "Russell 2000 Index",
    # "^CPI": "Comsumer Price Index",
}
tickers = list(indexes.keys())
# df = yf.download(tickers, start=start_date, end=end_date, interval='1mo')

prices = yf.download(tickers, start=start_date, end=end_date, interval="1d")["Close"]
# prices = yf.download("^GSPC", start="1929-01-01", interval="1d")

# Convert to monthly (end-of-month)
prices = prices.resample("MS").first()
prices.head()

[*********************100%***********************]  1 of 1 completed


Ticker,^GSPC
Date,
1947-01-01,15.20
1947-02-01,15.80
1947-03-01,15.41
1947-04-01,15.23
1947-05-01,14.69


In [4]:
fred = Fred(api_key=fred_api_key)
cpi = fred.get_series("CPIAUCSL")
cpi.head()

1947-01-01    21.48
1947-02-01    21.62
1947-03-01    22.00
1947-04-01    22.00
1947-05-01    21.95
dtype: float64

In [5]:
pricesi = prices.merge(cpi.rename("cpi"), left_index=True, right_index=True)
pricesi.tail()

,^GSPC,cpi
2025-11-01,6851.970215,325.063
2025-12-01,6812.629883,326.031
2026-01-01,6858.470215,326.588
2026-02-01,6976.439941,327.460
2026-03-01,6881.620117,330.293


In [6]:
tmp = pricesi.reset_index()
tmp.rename(columns={"index": "month"}, inplace=True)

arrayi = tmp.to_records()
arrayi[0:3]

rec.array([(0, '1947-01-01T00:00:00.000000000', 15.19999981, 21.48),
           (1, '1947-02-01T00:00:00.000000000', 15.80000019, 21.62),
           (2, '1947-03-01T00:00:00.000000000', 15.40999985, 22.  )],
          dtype=[('index', '<i8'), ('month', '<M8[ns]'), ('^GSPC', '<f8'), ('cpi', '<f8')])

#### Test the process in open code

In [7]:
starting_balance = 300_000.0  # $300k
time_til_event = 20 * 12  # 20 years (mike is 73)
duration_of_event = 5 * 12  # 5 years
start_index = 0
index_used = "^GSPC"
monthly_care_amount = 10_000.0  # Using a fixed amount for now
tax_rate = 0.25

# Ages at beginning
ti, mike = 54, 53

balance_at_event = (
    starting_balance
    * arrayi[start_index + time_til_event][index_used]
    / arrayi[start_index][index_used]
)
start_date = arrayi[start_index]["month"].astype("datetime64[D]")
event_date = arrayi[start_index + time_til_event]["month"].astype("datetime64[D]")
sb = "${:,.2f}".format(starting_balance)

print(f"Simulation begins on {start_date}, with initial investment of {sb}")
print(f"Event occurs on {event_date}")
print("Amount at event ${:,.2f}".format(balance_at_event))
print(f"Ages at event: Ti {ti+time_til_event/12}, Mike {mike+time_til_event/12}")

Simulation begins on 1947-01-01, with initial investment of $300,000.00
Event occurs on 1967-01-01
Amount at event $1,586,447.33
Ages at event: Ti 74.0, Mike 73.0


In [8]:
# Loop through months to manage payments
balance = balance_at_event
cum_spend, cum_withdraw, cum_tax = 0, 0, 0
out_df = pd.DataFrame(
    {
        "index": [time_til_event],
        "month": [event_date],
        "balance": [balance],
        "cum_spend": [cum_spend],
        "cum_withdraw": [cum_withdraw],
        "cum_tax": [cum_tax],
        "monthly_return(%)": [0.0],
    }
)


for m in range(time_til_event, time_til_event + duration_of_event + 1):
    # for m in range(time_til_event, time_til_event + 24 + 1):
    date = arrayi[m]["month"].astype("datetime64[D]")
    # spend = monthly_care_amount
    spend = monthly_care_amount * (arrayi[m]["cpi"] / arrayi[start_index]["cpi"])
    tax = monthly_care_amount * tax_rate
    withdraw = spend + tax

    cum_spend += spend
    cum_withdraw += withdraw
    cum_tax += tax

    balance -= withdraw
    monthly_return = arrayi[m][index_used] / arrayi[m - 1][index_used]
    balance = balance * monthly_return
    fbalance = "${:,.2f}".format(balance)
    formatted = [f"{x:,.2f}" for x in [balance, cum_spend, cum_withdraw, cum_tax]]

    print(m, date, formatted)

    new_row = {
        "index": m + 1,
        "month": date,
        "balance": balance,
        "cum_spend": cum_spend,
        "cum_withdraw": cum_withdraw,
        "cum_tax": cum_tax,
        "monthly_return(%)": 100 * (monthly_return - 1),
    }
    out_df.loc[len(out_df)] = new_row
    # print(m, date, fbalance, f"${cum_spend:,.2f}", cum_withdraw, cum_tax, f"{monthly_return:,.2f}",  "{:,.4f}".format(monthly_return-1))

240 1967-01-01 ['1,574,507.16', '15,316.57', '17,816.57', '2,500.00']
241 1967-02-01 ['1,673,808.77', '30,679.70', '35,679.70', '5,000.00']
242 1967-03-01 ['1,679,894.87', '46,042.83', '53,542.83', '7,500.00']
243 1967-04-01 ['1,691,555.13', '61,452.51', '71,452.51', '10,000.00']
244 1967-05-01 ['1,759,915.81', '76,862.20', '89,362.20', '12,500.00']
245 1967-06-01 ['1,674,902.21', '92,364.99', '107,364.99', '15,000.00']
246 1967-07-01 ['1,669,339.40', '107,914.34', '125,414.34', '17,500.00']
247 1967-08-01 ['1,732,252.67', '123,510.24', '143,510.24', '20,000.00']
248 1967-09-01 ['1,683,735.35', '139,152.70', '161,652.70', '22,500.00']
249 1967-10-01 ['1,712,483.16', '154,841.71', '179,841.71', '25,000.00']
250 1967-11-01 ['1,630,703.67', '170,623.84', '198,123.84', '27,500.00']
251 1967-12-01 ['1,643,505.96', '186,452.51', '216,452.51', '30,000.00']
252 1968-01-01 ['1,652,818.15', '202,327.75', '234,827.75', '32,500.00']
253 1968-02-01 ['1,574,026.87', '218,249.53', '253,249.53', '35,0

In [9]:
pd.options.display.float_format = "{:,.2f}".format
out_df.tail(36)

,index,month,balance,cum_spend,cum_withdraw,cum_tax,monthly_return(%)
26,266,1969-02-01,"1,514,058.30","414,106.15","479,106.15","65,000.00",-1.00
27,267,1969-03-01,"1,429,232.14","430,912.48","498,412.48","67,500.00",-4.38
28,268,1969-04-01,"1,453,397.38","447,811.92","517,811.92","70,000.00",3.09
29,269,1969-05-01,"1,463,501.41","464,757.91","537,257.91","72,500.00",2.06
30,270,1969-06-01,"1,436,010.82","481,797.02","556,797.02","75,000.00",-0.55
31,271,1969-07-01,"1,349,508.58","498,929.24","576,429.24","77,500.00",-4.72
32,272,1969-08-01,"1,267,324.55","516,108.01","596,108.01","80,000.00",-4.70
33,273,1969-09-01,"1,275,181.14","533,379.89","615,879.89","82,500.00",2.21
34,274,1969-10-01,"1,215,635.81","550,744.88","635,744.88","85,000.00",-3.16
35,275,1969-11-01,"1,255,513.35","568,202.98","655,702.98","87,500.00",5.00


##### Move process to a function

In [10]:
def run_trial(
    starting_balance,
    time_til_event,
    duration_of_event,
    start_index,
    monthly_care_amount,
    index_used="^GSPC",
    tax_rate=0.25,
    verbose=False,
    ti=ti,
    mike=mike,
    inflation_adj=1.0,
):
    balance_at_event = (
        starting_balance
        * arrayi[start_index + time_til_event][index_used]
        / arrayi[start_index][index_used]
    )
    start_date = arrayi[start_index]["month"].astype("datetime64[D]")
    event_date = arrayi[start_index + time_til_event]["month"].astype("datetime64[D]")
    sb = "${:,.2f}".format(starting_balance)
    mca = monthly_care_amount * (
        arrayi[time_til_event]["cpi"] / arrayi[start_index]["cpi"]
    )

    if verbose:
        print(
            f"Simulation begins on {start_date}, with initial investment of {sb} and monthly real care cost of {monthly_care_amount}"
        )
        print(f"    Event occurs on {event_date}")
        print(f"    Monthly care at Event", "${:,.2f}".format(mca))
        print("    Balance at event ${:,.2f}".format(balance_at_event))
        ti, mike = ti + time_til_event / 12, mike + time_til_event / 12
        print(f"    Ages at event: Ti {math.floor(ti)}, Mike {math.floor(mike)}")

    # Loop through months to manage payments
    balance = balance_at_event
    cum_spend, cum_withdraw, cum_tax = 0, 0, 0
    out_df = pd.DataFrame(
        {
            "index": [time_til_event],
            "month": [event_date],
            "balance": [balance],
            "cum_spend": [cum_spend],
            "cum_withdraw": [cum_withdraw],
            "cum_tax": [cum_tax],
            "monthly_return(%)": [0.0],
        }
    )

    for m in range(time_til_event, time_til_event + duration_of_event + 1):
        # for m in range(time_til_event, time_til_event + 24 + 1):
        date = arrayi[m]["month"].astype("datetime64[D]")

        # Add inflation to monthly care
        spend = monthly_care_amount * (arrayi[m]["cpi"] / arrayi[start_index]["cpi"])
        tax = spend * tax_rate
        withdraw = spend + tax

        ti += 1 / 12
        mike += 1 / 12
        cum_spend += spend
        cum_withdraw += withdraw
        cum_tax += tax
        balance -= withdraw
        monthly_return = arrayi[m][index_used] / arrayi[m - 1][index_used]
        balance = balance * monthly_return
        fbalance = "${:,.2f}".format(balance)
        formatted = [f"{x:,.2f}" for x in [balance, cum_spend, cum_withdraw, cum_tax]]

        # print(m, date, formatted)

        new_row = {
            "index": m + 1,
            "month": date,
            "balance": balance,
            "cum_spend": cum_spend,
            "cum_withdraw": cum_withdraw,
            "cum_tax": cum_tax,
            "monthly_return(%)": 100 * (monthly_return - 1),
        }
        out_df.loc[len(out_df)] = new_row

    new_row = pd.DataFrame(
        {
            "start_index": [start_index],
            "start_date": [start_date],
            "event_date": [event_date],
            "time_til_event": [time_til_event],
            "mike_age": [math.floor(mike)],
            "inflation_adj": inflation_adj,
            "starting_balance": [starting_balance],
            "index_used": [index_used],
            "duration_of_event": [duration_of_event],
            "starting_care_amount": [monthly_care_amount],
            "cum_spend": [cum_spend],
            "cum_tax": [cum_tax],
            "cum_withdraw": [cum_withdraw],
            "ending_balance": [balance],
        }
    )

    if verbose:
        print("\nEnd of event:")
        print(f"    Ages at end: Ti {math.floor(ti)}, Mike {math.floor(mike)}")
        print(f"    Cum Spend             ", "${:,.2f}".format(cum_spend))
        print(f"    Cum Tax               ", "${:,.2f}".format(cum_tax))
        print(f"    Cum Withdraws         ", "${:,.2f}".format(cum_withdraw))
        print(f"    Remaining balance of   {fbalance}")

    return new_row

In [11]:
starting_balance = 300_000.0
time_til_event = 30 * 12
duration_of_event = 5 * 12
start_index = 0
index_used = "^GSPC"
monthly_care_amount = 7_500.0
tax_rate = 0.25
inflation_adj = 1.1

# Ages at beginning
ti, mike = 54, 53

run_trial(
    starting_balance,
    time_til_event,
    duration_of_event,
    start_index,
    monthly_care_amount,
    index_used="^GSPC",
    tax_rate=0.25,
    verbose=True,
    inflation_adj=1.0,
)
run_trial(
    starting_balance,
    time_til_event,
    duration_of_event,
    start_index,
    monthly_care_amount,
    index_used="^GSPC",
    tax_rate=0.25,
    verbose=True,
    inflation_adj=1.1,
)

Simulation begins on 1947-01-01, with initial investment of $300,000.00 and monthly real care cost of 7500.0
    Event occurs on 1977-01-01
    Monthly care at Event $20,495.81
    Balance at event $2,111,842.13
    Ages at event: Ti 84, Mike 83

End of event:
    Ages at end: Ti 89, Mike 88
    Cum Spend              $1,590,607.54
    Cum Tax                $397,651.89
    Cum Withdraws          $1,988,259.43
    Remaining balance of   $280,555.09
Simulation begins on 1947-01-01, with initial investment of $300,000.00 and monthly real care cost of 7500.0
    Event occurs on 1977-01-01
    Monthly care at Event $20,495.81
    Balance at event $2,111,842.13
    Ages at event: Ti 84, Mike 83

End of event:
    Ages at end: Ti 89, Mike 88
    Cum Spend              $1,590,607.54
    Cum Tax                $397,651.89
    Cum Withdraws          $1,988,259.43
    Remaining balance of   $280,555.09


,start_index,start_date,event_date,time_til_event,mike_age,inflation_adj,starting_balance,index_used,duration_of_event,starting_care_amount,cum_spend,cum_tax,cum_withdraw,ending_balance
0,0,1947-01-01,1977-01-01,360,88,1.10,"300,000.00",^GSPC,60,"7,500.00","1,590,607.54","397,651.89","1,988,259.43","280,555.09"


In [12]:
starting_balance = 300_000.0
event_times = [10 * 12, 15 * 12]
durations = [2 * 12, 5 * 12]
starts = [
    0,
    10,
]
trials = pd.DataFrame()

for time_til_event in event_times:
    for duration_of_event in durations:
        for start_index in starts:
            trial = run_trial(
                starting_balance,
                time_til_event,
                duration_of_event,
                start_index,
                monthly_care_amount,
                index_used="^GSPC",
                tax_rate=0.25,
            )
            trials = pd.concat([trials, trial])
trials.head(10)

,start_index,start_date,event_date,time_til_event,mike_age,inflation_adj,starting_balance,index_used,duration_of_event,starting_care_amount,cum_spend,cum_tax,cum_withdraw,ending_balance
0,0,1947-01-01,1957-01-01,120,55,1.00,"300,000.00",^GSPC,24,"7,500.00","248,931.56","62,232.89","311,164.46","716,416.20"
0,10,1947-11-01,1957-11-01,120,55,1.00,"300,000.00",^GSPC,24,"7,500.00","231,875.54","57,968.89","289,844.43","589,401.58"
0,0,1947-01-01,1957-01-01,120,58,1.00,"300,000.00",^GSPC,60,"7,500.00","620,673.88","155,168.47","775,842.35","360,988.86"
0,10,1947-11-01,1957-11-01,120,58,1.00,"300,000.00",^GSPC,60,"7,500.00","578,147.22","144,536.81","722,684.03","236,511.69"
0,0,1947-01-01,1962-01-01,180,55,1.00,"300,000.00",^GSPC,24,"7,500.00","265,914.80","66,478.70","332,393.51","1,089,829.21"
0,10,1947-11-01,1962-11-01,180,55,1.00,"300,000.00",^GSPC,24,"7,500.00","247,695.14","61,923.79","309,618.93","809,025.08"
0,0,1947-01-01,1962-01-01,180,58,1.00,"300,000.00",^GSPC,60,"7,500.00","664,800.98","166,200.24","831,001.22","686,287.85"
0,10,1947-11-01,1962-11-01,180,58,1.00,"300,000.00",^GSPC,60,"7,500.00","619,250.87","154,812.72","774,063.58","419,605.95"


In [13]:
starting_balance = 300_000.0
monthly_care_amount = 8000.0
event_times = [10, 15, 20, 25, 30]
event_times = [x * 12 for x in event_times]
durations = [2 * 12, 5 * 12, 10 * 12]
starts = range(0, 480, 12)
inflation_adj = 1.0

trials = pd.DataFrame()

for time_til_event in event_times:
    for duration_of_event in durations:
        for start_index in starts:
            trial = run_trial(
                starting_balance,
                time_til_event,
                duration_of_event,
                start_index,
                monthly_care_amount,
                index_used="^GSPC",
                tax_rate=0.25,
            )
            trials = pd.concat([trials, trial])
trials = trials.reset_index()
trials.head()

,index,start_index,start_date,event_date,time_til_event,mike_age,inflation_adj,starting_balance,index_used,duration_of_event,starting_care_amount,cum_spend,cum_tax,cum_withdraw,ending_balance
0,0,0,1947-01-01,1957-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","265,527.00","66,381.75","331,908.75","690,880.88"
1,0,12,1948-01-01,1958-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","240,858.11","60,214.53","301,072.64","580,388.11"
2,0,24,1949-01-01,1959-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","237,547.69","59,386.92","296,934.61","975,884.12"
3,0,36,1950-01-01,1960-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","242,599.74","60,649.94","303,249.68","927,480.89"
4,0,48,1951-01-01,1961-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","224,724.98","56,181.25","280,906.23","656,834.12"


In [14]:
trials.tail()

,index,start_index,start_date,event_date,time_til_event,mike_age,inflation_adj,starting_balance,index_used,duration_of_event,starting_care_amount,cum_spend,cum_tax,cum_withdraw,ending_balance
595,0,420,1982-01-01,2012-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","913,610.17","228,402.54","1,142,012.71","5,422,589.29"
596,0,432,1983-01-01,2013-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","880,947.91","220,236.98","1,101,184.88","5,617,251.35"
597,0,444,1984-01-01,2014-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","844,709.11","211,177.28","1,055,886.39","6,130,321.27"
598,0,456,1985-01-01,2015-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","815,939.45","203,984.86","1,019,924.31","7,117,957.03"
599,0,468,1986-01-01,2016-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","784,757.05","196,189.26","980,946.31","5,138,018.28"


In [15]:
# Write to an Excel file with multiple tabs
trials.to_excel("trials.xlsx")

In [16]:
event_times = [10, 15, 20, 25, 30]
event_times = [x * 12 for x in event_times]
print(event_times)

[120, 180, 240, 300, 360]


In [17]:
starting_balance = 450_000.0
monthly_care_amount = 8000.0
durations = [5 * 12]
inflation_adj = 1.0

event_times = [x * 12 for x in [10, 15, 20, 25, 30]]
event_times = range(0, 12 * 30, 24)
starts = range(0, 600, 24)

trials = pd.DataFrame()

for time_til_event in event_times:
    for duration_of_event in durations:
        for start_index in starts:
            trial = run_trial(
                starting_balance,
                time_til_event,
                duration_of_event,
                start_index,
                monthly_care_amount,
                index_used="^GSPC",
                tax_rate=0.20,
            )
            trials = pd.concat([trials, trial])
trials = trials.reset_index()

# Create pivot table
tmp = trials.copy(deep=True)
tmp["time_til_event"] = round(tmp["time_til_event"] / 12)
tmp["ending_balance"] = tmp["ending_balance"] / 1000

pivot_df = pd.pivot_table(
    tmp, index="start_date", columns="time_til_event", values="ending_balance"
)
v = np.nanmax(np.abs(pivot_df.values))

styled = pivot_df.style.background_gradient(cmap="RdYlGn", vmin=-v, vmax=v).format(
    "{:,.0f}"
)
styled

time_til_event,0.000000,2.000000,4.000000,6.000000,8.000000,10.000000,12.000000,14.000000,16.000000,18.000000,20.000000,22.000000,24.000000,26.000000,28.000000
start_date,,,,,,,,,,,,,,,
1947-01-01 00:00:00,-905,-119,189,372,921,"1,039","1,349","1,764","1,858","1,841","1,920","1,684","1,582","1,369","1,362"
1949-01-01 00:00:00,-810,275,740,966,"1,560","1,617","1,585","2,182","3,020","1,856","2,987","1,498","2,632",393,"3,331"
1951-01-01 00:00:00,-766,233,793,843,"1,274","1,018","1,097","2,131","1,837","1,715","1,575","1,404",547,627,"1,675"
1953-01-01 00:00:00,-730,347,813,782,917,777,"1,238","1,404","1,910",966,"1,676",192,900,142,"2,085"
1955-01-01 00:00:00,-726,255,577,416,553,732,606,"1,257",912,873,208,304,250,172,"1,133"
1957-01-01 00:00:00,-702,234,338,290,673,377,672,644,"1,007",35,468,0,431,-48,"1,052"
1959-01-01 00:00:00,-670,144,263,436,412,509,318,814,128,274,181,188,262,-0,"1,724"
1961-01-01 00:00:00,-651,200,631,366,715,328,612,140,562,168,560,188,464,497,"1,998"
1963-01-01 00:00:00,-638,402,462,564,450,563,16,520,356,429,484,299,976,557,"2,322"


In [20]:
starting_balance = 350_000.0
monthly_care_amount = 8000.0
durations = [5 * 12]
inflation_adj = 1.0

event_times = [x * 12 for x in [10, 15, 20, 25, 30]]
event_times = range(0, 12 * 30, 24)
starts = range(0, 600, 24)

trials = pd.DataFrame()

for time_til_event in event_times:
    for duration_of_event in durations:
        for start_index in starts:
            trial = run_trial(
                starting_balance,
                time_til_event,
                duration_of_event,
                start_index,
                monthly_care_amount,
                index_used="^GSPC",
                tax_rate=0.20,
            )
            trials = pd.concat([trials, trial])
trials = trials.reset_index()

# Create pivot table
tmp = trials.copy(deep=True)
tmp["time_til_event"] = round(tmp["time_til_event"] / 12)
tmp["ending_balance"] = tmp["ending_balance"] / 1000

pivot_df = pd.pivot_table(
    tmp, index="start_date", columns="time_til_event", values="ending_balance"
)
v = np.nanmax(np.abs(pivot_df.values))

styled = pivot_df.style.background_gradient(cmap="RdYlGn", vmin=-v, vmax=v).format(
    "{:,.0f}"
)

styled

time_til_event,0.000000,2.000000,4.000000,6.000000,8.000000,10.000000,12.000000,14.000000,16.000000,18.000000,20.000000,22.000000,24.000000,26.000000,28.000000
start_date,,,,,,,,,,,,,,,
1947-01-01 00:00:00,-905,-282,-125,98,495,570,827,"1,133","1,218","1,224","1,249","1,066",959,743,645
1949-01-01 00:00:00,-810,44,332,580,"1,015","1,045","1,034","1,483","2,146","1,258","2,104",948,"1,804",18,"2,220"
1951-01-01 00:00:00,-766,20,387,493,803,590,665,"1,455","1,237","1,158","1,018",886,196,215,951
1953-01-01 00:00:00,-730,117,413,453,535,412,783,899,"1,303",584,"1,106",-47,481,-149,"1,287"
1955-01-01 00:00:00,-726,46,231,170,253,378,293,786,527,512,-34,41,-23,-125,548
1957-01-01 00:00:00,-703,35,52,77,351,108,350,315,607,-134,174,-189,125,-287,496
1959-01-01 00:00:00,-670,-28,3,197,157,219,83,456,-69,59,-40,-34,3,-239,"1,034"
1961-01-01 00:00:00,-651,19,295,147,396,84,316,-63,274,-19,260,-29,165,154,"1,255"
1963-01-01 00:00:00,-639,179,167,303,194,270,-145,236,117,187,204,61,568,206,"1,513"
